<a href="https://colab.research.google.com/github/Dan2222001/Pfizer-Externship-AI-Powered-PDF-Reader/blob/main/Pfizer_Externship_AI_Powered_PDF_Reader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Installs and Imports

In [ ]:
# Remove Colab's preinstalled GPU PyTorch
# PyTorch will be kept on CPU to avoid conflicts with Paddle's CUDA libraries
!pip uninstall -y torch torchvision torchaudio

# PyTorch - CPU only
!pip install --no-cache-dir -q \
    torch==2.11.0 \
    torchvision==0.26.0 \
    torchaudio==2.11.0 \
    --index-url https://download.pytorch.org/whl/cpu


# Core RAG / UI
!pip install -q llama-index
!pip install -q llama-index-readers-file
!pip install -q llama-index-embeddings-huggingface
!pip install -q langchain-text-splitters
!pip install -q gradio
!pip install -q pdf2image
!pip install -q llama-index-retrievers-bm25


# Mistral / llama.cpp
!pip install -q llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124


# PaddleOCR - GPU
!pip install --no-cache-dir -q \
    paddlepaddle-gpu==3.3.0 \
    -i https://www.paddlepaddle.org.cn/packages/stable/cu130/

!pip install --no-cache-dir -q paddleocr==3.7.0


# PDF image conversion
!apt-get update -qq
!apt-get install -y -qq poppler-utils

Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.3/190.3 MB 282.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 334.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.3/341.3 kB 354.4 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
#imports

import os

os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["OMP_NUM_THREADS"] = "1"

# Keep JAX/BM25 off the GPU
os.environ["JAX_PLATFORMS"] = "cpu"

import time
import json
import copy
import gc

import numpy as np
import gradio as gr
import paddle

from paddleocr import PaddleOCR
from pdf2image import convert_from_path

from llama_cpp import Llama

from llama_index.readers.file import PDFReader
from llama_index.core import Document as LlamaDocument
from llama_index.core import VectorStoreIndex, Settings

from llama_index.core.vector_stores.types import (
    MetadataFilters,
    MetadataFilter,
    FilterOperator
)

from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.retrievers.bm25 import BM25Retriever

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# Check PaddleOCR GPU setup
print("Paddle version:", paddle.__version__)
print("Compiled with CUDA:", paddle.is_compiled_with_cuda())

paddle.device.set_device("gpu:0")

print("Current Paddle device:", paddle.device.get_device())

Paddle version: 3.3.0
Compiled with CUDA: True
Current Paddle device: gpu:0


#Mistral Setup

In [ ]:
# Download Mistral model if not already present
model_path = "/content/mistral.gguf"
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

# Verify file exists and check size
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")


#This should free up memory for the model setup
if "model" in globals():
    try:
        model.close()
    except:
        pass
    del model

if "ocr_engine" in globals():
    del ocr_engine

gc.collect()


try:
    paddle.device.cuda.empty_cache()
except:
    pass

!nvidia-smi

#Checks GPU usage for debugging memory issues
!nvidia-smi

#Setup model
model = Llama(
    model_path=model_path,
    n_ctx=8192, #Maximum context size in tokens for each Mistral request. Higher values allow larger prompts but use more memory.
    n_gpu_layers=-1,
    verbose=False
)

def mistral_model(prompt, system_prompt="Follow the user's instructions carefully.", max_tokens=700):
    response = model.create_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=0.1
    )

    return response["choices"][0]["message"]["content"]

Model file exists. Size: 4166.07 MB
Thu Aug 20 04:31:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             42W /   70W |     157MiB /  15360MiB |      1%      Default |
|                                         |                        |                  N/A |
+-----------

#Doc Tagging and Query Functions (And a function to serialize the results)

In [ ]:
doc_tags = [] #instead of giving each doc a single category, it gets multiple tags for the information in it. this better represents each doc

# Converts retrieved llm tags into a list
def clean_llm_tags(response):
    cleaned = (response.strip().replace('"', '').replace('`', '').replace('*', '').lower())

    tags = [
        tag.strip()
        for tag in cleaned.split(",")
        if tag.strip()
    ]

    return tags

# Converts retrieved page numbers into a list
def clean_page_numbers(response):
    cleaned = (response.strip().replace('"', '').replace('`', '').replace(' ', ''))

    pages = []

    for value in cleaned.split(","):
        if value.isdigit():
            page = int(value)

            # Avoid duplicate page numbers
            if page not in pages:
                pages.append(page)

    return pages


# Decides which labels the query matches with
def classify_query_llm(query, metadata_store):
    total_pages = len(metadata_store)

    #scales number of pages to select to total number of pages, with a min of 3 and max of 8, unless the PDF is under 3 pages
    num_pages_to_select = min(total_pages, max(3, round(total_pages * 0.1)), 8)

    doc_list = "\n".join(
        [
            f"Page {doc['page']}: tags: {','.join(doc['doc_tags'])} - summary: {doc['routing_summary']}"
            for doc in metadata_store
        ]
    )

    prompt = f"""
You are an intelligent assistant that routes user queries to the most relevant
pharmaceutical document pages.

Available pages:
{doc_list}

User query: "{query}"

Select the {num_pages_to_select} pages most likely to contain information needed to answer
the user's query.

Consider the tags and routing summary associated with each page.

Pay particular attention to specific names, identifiers, document titles,
section headings, codes, dates, formulations, products, organizations, or
other entities mentioned in the user's query.

A page that matches both the specific subject/entity and the requested topic
should normally rank above a page that only matches the general topic.

Strongly deprioritize a page if its summary clearly refers to a different
specific subject or entity than the one requested.

Table-of-contents, index, and overview pages should rank below pages containing
the actual requested information when such pages are available.

Select pages based on how likely they are to contain the actual answer,
not just related information.

Rules:
- Return exactly {num_pages_to_select} page numbers.
- Rank them from most likely to least likely to contain the answer.
- Do not repeat a page number.
- Respond ONLY with the page numbers separated by commas.
- Do not include spaces, explanations, or any other text.

Example (for 3 or more pages):
3,5,2
"""

    response = mistral_model(prompt, max_tokens=64) #use less tokens for routing
    print(f"Raw routing response: {response}") #prints routing response for debugging
    selected_pages = clean_page_numbers(response)
    selected_pages = [page for page in selected_pages if 1 <= page <= total_pages]
    return selected_pages[:num_pages_to_select]

In [ ]:
# Labels docs and creates a short summary for query routing
def classify_doc_tags_llm(text, max_chars=2500):
    truncated_text = text[:max_chars]

    prompt = f"""
You are analyzing a pharmaceutical document page for later query routing.

Used tags:
\"\"\"
{doc_tags}
\"\"\"

Document content:
\"\"\"
{truncated_text}
\"\"\"

Return:
TAGS: up to 8 concise information-type tags
SUMMARY: a routing summary of no more than 40 words

Rules:
- Prioritize the page's main subjects and information types.
- Include important names, identifiers, document types, or section topics when they help distinguish this page from others.
- Do not create tags for every incidental word, chemical, number, or minor detail.
- Use existing tags whenever appropriate.
- Create new tags only when necessary.
- New tags should be lowercase snake_case.
- The summary should identify the main subject, entity, product, document, organization, or identifier when present.
- The summary should state the main topics covered and important conclusions that could help answer a query.
- Identify table-of-contents, index, or overview pages as such.
- Respond in exactly two lines.
- Do not include any other text.

Example:
TAGS: tag_1,tag_2,tag_3
SUMMARY: Short description of the specific information contained on this page.
"""

    try:
        response = mistral_model(prompt, max_tokens=160)

        lines = response.strip().splitlines()

        tags_line = next((line for line in lines if line.lower().startswith("tags:")), "")
        summary_line = next((line for line in lines if line.lower().startswith("summary:")), "")

        tags = clean_llm_tags(tags_line.split(":", 1)[1])[:8] if tags_line else ["unknown"]
        summary = summary_line.split(":", 1)[1].strip() if summary_line else " ".join(truncated_text.split())[:300]

        return tags, summary

    except Exception as e:
        print("LLM failed:", e)
        return ["unknown"], " ".join(truncated_text.split())[:300]

In [ ]:
#Serializes the RAG data for printing as json
def serialize_results(results):
    return [
        {
            "rank": rank,
            "text": result.node.get_content(),
            "metadata": result.node.metadata,
            "score": result.score
        }
        for rank, result in enumerate(results, start=1)
    ]

#PDF Processing

In [ ]:
rag_data = { #saving this data outside of functions so all can use it
    "metadata_store": None,
    "index": None,
    "json_info": None,
    "documents": None,
    "query_times": [],
    "retrieval_times": [],
    "llm_times": [],
    "routing_times": []
}

# Converts uploaded PDF into chunks with metadata, including category labels
def process_pdf(pdf_path):
    pdf_start_time = time.perf_counter() #used to record PDF processing time

    #Clearing rag data for new PDF
    global doc_tags

    doc_tags.clear()
    rag_data["metadata_store"] = None
    rag_data["index"] = None
    rag_data["json_info"] = None
    rag_data["documents"] = None
    rag_data["query_times"] = []
    rag_data["retrieval_times"] = []
    rag_data["llm_times"] = []
    rag_data["routing_times"] = []

    metadata_store = []

    if pdf_path is None:
        return "Please upload a PDF."

    loader = PDFReader()
    documents = loader.load_data(pdf_path)

    if not documents:
        return "No readable pages were found."

    #Checking if the PDF is scanned or not
    sample_text = "".join(
        document.text or ""
        for document in documents[:3]
    ).strip()

    if len(sample_text) < 100:
        print("Scanned PDF detected. Running OCR.")
        documents = load_scanned_pdf(pdf_path)
    else:
        print("Text-based PDF detected.")

    #Adds data for each doc
    for page_index, page_document in enumerate(documents):
        metadata_store.append(
            {
                "page": page_index + 1,
                "text": page_document.text,
                "doc_tags": [],
                "routing_summary": "",
                "source_file": os.path.basename(page_document.metadata.get("source_file",pdf_path))
            }
        )


    for page_index, page in enumerate(metadata_store):
        print(f"Tagging page {page_index + 1}...")

        page["doc_tags"], page["routing_summary"] = classify_doc_tags_llm(
            page["text"]
        )

        # Add any newly-created tags to the global tag list
        for tag in page["doc_tags"]:
            if tag not in doc_tags:
                doc_tags.append(tag)

        time.sleep(1)

    for page in metadata_store:
        print(f"Page {page['page']} | Tags: {page['doc_tags']}")
        print(f"Summary: {page['routing_summary']}")

    #Recursive chunking
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=512,
        chunk_overlap=100
    )

    sliding_window_documents = []

    for page in metadata_store:

        chunks = splitter.split_text(
            page["text"]
        )

        for chunk_index, chunk in enumerate(chunks):
            sliding_window_documents.append(
                LlamaDocument(
                    text=chunk,
                        metadata={
                            "doc_tags": page["doc_tags"],
                            "chunk_index": chunk_index,
                            "source_file": page["source_file"],
                            "page": page["page"],
                            "page_start": page["page"],
                            "page_end": page["page"]
                        }
                )
            )



    Settings.embed_model = HuggingFaceEmbedding(
        model_name="BAAI/bge-small-en-v1.5"
    )

    sliding_window_index = VectorStoreIndex.from_documents(
        sliding_window_documents
    )

    print(
        f"Sliding-window index: "
        f"{len(sliding_window_documents)} chunks"
    )

    #Storing data for later
    rag_data["metadata_store"] = metadata_store
    rag_data["index"] = sliding_window_index
    rag_data["documents"] = sliding_window_documents

    #print processing time
    total_pdf_time = time.perf_counter() - pdf_start_time
    print(f"\nTotal PDF processing time: {total_pdf_time:.2f} seconds")

    return (
        f"Ready: processed {len(documents)} pages and created "
        f"{len(sliding_window_documents)} chunks."
    )

#OCR Setup

In [ ]:
ocr_engine = PaddleOCR(
    use_textline_orientation=True,
    lang="en",
    device="gpu:0"
)

# Function to extract text from a page with OCR
def extract_text_with_paddleocr(image):
    image_array = np.array(image.convert("RGB"))

    results = ocr_engine.predict(image_array)

    extracted_lines = []

    for result in results:
        texts = result["rec_texts"]
        scores = result["rec_scores"]

        for text, confidence in zip(texts, scores):
            if text and confidence >= 0.50:
                extracted_lines.append(text)

    return "\n".join(extracted_lines)

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP

In [ ]:
#Uses OCR function to get information from scanned documents
def load_scanned_pdf(pdf_path, dpi=300):
    images = convert_from_path(pdf_path, dpi=dpi)
    documents = []

    source_file = os.path.basename(pdf_path)

    for page_index, image in enumerate(images):
        page_number = page_index + 1

        start_time = time.perf_counter() #tracks time for OCR

        text = extract_text_with_paddleocr(image)

        elapsed_time = time.perf_counter() - start_time

        documents.append(
            LlamaDocument(
                text=text,
                metadata={
                    "source_file": source_file,
                    "page": page_number
                }
            )
        )

        print(f"\n--- OCR PAGE {page_number} ---")
        print(text)
        print(f"OCR time: {elapsed_time:.2f} seconds")

    return documents

#Query Routing and Hybrid Retrieval - Page by Page

In [ ]:
# Combines results from BM25 and vector retrieval for hybrid retrieval
def reciprocal_rank_fusion(
    vector_results,
    bm25_results,
    top_k=4,
    k=60
):
    scores = {}
    nodes = {}

    for results in [vector_results, bm25_results]:
        for rank, result in enumerate(results, start=1):
            metadata = result.node.metadata

            #prevents duplicate chunks from being retrieved if both results have the same chunk
            node_id = (
                metadata.get("source_file"),
                metadata.get("page"),
                metadata.get("chunk_index")
            )

            if node_id not in scores:
                scores[node_id] = 0
                nodes[node_id] = result

            scores[node_id] += 1 / (k + rank)

    ranked_ids = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    final_results = []

    for node_id in ranked_ids[:top_k]:
        result = nodes[node_id]
        result.score = scores[node_id]
        final_results.append(result)

    return final_results



# Reads user query to find answers from document. Also stores RAG info in JSON format
def process_query(user_query, history):
    query_start_time = time.perf_counter() #used to record query processing time

    metadata_store = rag_data["metadata_store"]
    sliding_window_index = rag_data["index"]
    sliding_window_documents = rag_data["documents"]

    if metadata_store is None or sliding_window_index is None or sliding_window_documents is None:
        return "Please upload and process a PDF first."

    if not user_query or not user_query.strip():
        return "Please enter a question."


    #Using query routing to find relevant documents
    routing_start_time = time.perf_counter()

    selected_pages = classify_query_llm(
        user_query,
        metadata_store
    )

    routing_time = time.perf_counter() - routing_start_time

    print(f"\nSelected pages: {selected_pages}")
    print("Page routing ranking:")

    for rank, page in enumerate(selected_pages, start=1):
        print(f"Rank {rank}: Page {page}")

    retrieval_start_time = time.perf_counter()

    metadata_filter = MetadataFilters(
        filters=[
            MetadataFilter(
                key="page",
                value=selected_pages,
                operator=FilterOperator.IN
            )
        ]
    )

    #Scales number of chunks to retrieve before fusion to page number, with a min of 6 and max of 16. This does NOT scale the number of chunks after fusion
    num_candidates = min(max(6, len(selected_pages) * 2), 16)

    # Vector retrieval from the selected pages
    vector_retriever = sliding_window_index.as_retriever(
        similarity_top_k=num_candidates,
        filters=metadata_filter
    )

    vector_results = vector_retriever.retrieve(
        user_query
    )


    # Limit BM25 to chunks from the selected pages
    selected_documents = [
        document
        for document in sliding_window_documents
        if document.metadata["page"] in selected_pages
    ]


    # BM25 keyword retrieval
    bm25_retriever = BM25Retriever.from_defaults(
        nodes=selected_documents,
        similarity_top_k=num_candidates
    )

    bm25_results = bm25_retriever.retrieve(
        user_query
    )


    # Combine vector and BM25 results with top k = 4
    sliding_window_results = reciprocal_rank_fusion(
        vector_results,
        bm25_results,
        top_k=4
    )

    retrieval_time = time.perf_counter() - retrieval_start_time

    print("\n--- FINAL RETRIEVAL RANKING ---")

    for rank, result in enumerate(sliding_window_results, start=1):
        metadata = result.node.metadata
        page_start = metadata.get("page_start", metadata.get("page", "Unknown"))
        page_end = metadata.get("page_end", page_start)

        if page_start == page_end:
            page_label = f"Page {page_start}"
        else:
            page_label = f"Pages {page_start}-{page_end}"

        preview = result.node.get_content().replace("\n", " ")[:200]

        print(f"Rank {rank} | {page_label} | Fusion score: {result.score:.6f}")
        print(f"Preview: {preview}")

    chunk_count = len(sliding_window_results)

    #Handles when there are no chunks
    if chunk_count == 0:
        rag_data["json_info"] = json.dumps(
            {
                "query": user_query,
                "selected_pages": selected_pages,
                "average_fusion_score": 0.0,
                "chunk_count": 0,
                "hybrid_retrieval": {
                    "matched_chunks": [],
                    "answer": "No matching chunks were found."
                }
            },
            indent=2
        )

        return (
            f"No matching information was found on "
            f"the selected pages: {selected_pages}."
        )

    # Copy the results so the original indexed nodes are not modified
    citation_results = copy.deepcopy(sliding_window_results)

    # Add each chunk's citation metadata directly to its context
    for result in citation_results:
        metadata = result.node.metadata

        source_file = metadata.get(
            "source_file",
            "Unknown source"
        )

        page_start = metadata.get(
            "page_start",
            "Unknown page"
        )

        page_end = metadata.get(
            "page_end",
            page_start
        )

        if page_start == page_end:
            citation_label = (
                f"[Source: {source_file}, Page {page_start}]"
            )
        else:
            citation_label = (
                f"[Source: {source_file}, "
                f"Pages {page_start}-{page_end}]"
            )

        original_text = result.node.get_content()

        result.node.set_content(
            f"{citation_label}\n{original_text}"
        )

    context_text = "\n\n".join(
        result.node.get_content()
        for result in citation_results
    )

    #Query used to have the AI use retrived information to answer user query with citation
    citation_query = f"""
    Answer the following question using only the retrieved document context.

    Question:
    {user_query}

    Retrieved document context:
    {context_text}

    Instructions:
    - Use only the retrieved context.
    - Not all retrieved context needs to be used. Ignore context that is not directly relevant to the specific question being asked.
    - Do not combine information from different products/items unless the question explicitly asks for a comparison.
    - If retrieved context clearly refers to a different named subject, item, entity, or document than the one requested, do not attribute that information to the requested subject.
    - Do not use outside knowledge.
    - If the answer is absent, say it cannot be determined from the document.
    - Place the appropriate citation after each factual statement. If multiple consecutive statements use the same citation, only place the citation once for the last of the consecutive statements
    - Copy citation labels exactly.
    - Do not invent filenames, page numbers, or page ranges.
    - At the end, include a Sources section containing each source used.
    """

    #times mistral generation time
    llm_start_time = time.perf_counter()
    answer = mistral_model(
        citation_query,
        system_prompt="Answer using only the supplied document context. If the answer is not present, say that it cannot be determined from the document. Use the given source citations."
    )
    llm_time = time.perf_counter() - llm_start_time

    print("\n--- QUERY AND RESPONSE ---")
    print(f"Query: {user_query}")
    print(f"\nResponse:\n{answer}")

    #Averages all retrieval scores to return the final retrieval confidence score
    scores = [
        result.score
        for result in sliding_window_results
        if result.score is not None
    ]

    if scores:
        average_fusion_score = sum(scores) / len(scores)
    else:
        average_fusion_score = 0.0

    total_query_time = time.perf_counter() - query_start_time

    rag_data["query_times"].append(total_query_time)
    rag_data["retrieval_times"].append(retrieval_time)
    rag_data["llm_times"].append(llm_time)
    rag_data["routing_times"].append(routing_time)

    query_count = len(rag_data["query_times"])
    average_query_time = sum(rag_data["query_times"]) / query_count
    average_retrieval_time = sum(rag_data["retrieval_times"]) / query_count
    average_llm_time = sum(rag_data["llm_times"]) / query_count
    average_routing_time = sum(rag_data["routing_times"]) / query_count

    print("\n--- QUERY PERFORMANCE ---")
    print(f"Routing time: {routing_time:.2f} seconds")
    print(f"Retrieval latency: {retrieval_time:.3f} seconds ({retrieval_time * 1000:.0f} ms)")
    print(f"LLM generation time: {llm_time:.2f} seconds")
    print(f"Total response time: {total_query_time:.2f} seconds")

    print(f"\n--- RUNNING AVERAGES AFTER {query_count} QUERIES ---")
    print(f"Average routing time: {average_routing_time:.2f} seconds")
    print(f"Average retrieval latency: {average_retrieval_time:.3f} seconds ({average_retrieval_time * 1000:.0f} ms)")
    print(f"Average LLM generation time: {average_llm_time:.2f} seconds")
    print(f"Average response time: {average_query_time:.2f} seconds")

    comparison_output = {
        "query": user_query,
        "selected_pages": selected_pages,
        "average_fusion_score": average_fusion_score,
        "chunk_count": chunk_count,
        "hybrid_retrieval": {
            "matched_chunks": serialize_results(sliding_window_results),
            "answer": answer
        },
        "timings": {
            "routing_seconds": routing_time,
            "retrieval_seconds": retrieval_time,
            "llm_generation_seconds": llm_time,
            "total_response_seconds": total_query_time
        }
    }

    #Saves data in json format for downloading later
    rag_data["json_info"] = (
        json.dumps(
            comparison_output,
            indent=2
        )
    )


    #prints total query processing time
    total_time = time.perf_counter() - query_start_time
    print(f"\nTotal query processing time: {total_time:.2f} seconds")

    return (
        f"**Selected pages:** `{selected_pages}`\n\n"
        f"**Average RRF score:** {average_fusion_score:.4f}\n\n"
        f"**Chunks retrieved:** {chunk_count}\n\n"
        f"{answer}"
    )

#Print Chunk Data to json

In [ ]:
#This function was an optional part I added to a previous assignment. Since I used that as a base, I've kept it here
def create_json_download():
    json_text = rag_data["json_info"]

    if json_text is None:
        raise gr.Error("No chunk data yet. Ask a question first.")

    output_path = "/content/rag_results.txt"

    with open(output_path, "w", encoding="utf-8") as file:
        file.write(json_text)

    return output_path

#Gradio Frontend

In [ ]:
#Checks if the GPU is being used. Only needed for bug testing
!nvidia-smi

Thu Aug 20 04:31:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             29W /   70W |    6033MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#UI for uploading/downloading files and submitting queries
with gr.Blocks(title="PDF Reader With Routing") as demo:
    gr.Markdown(
        """
        # PDF Reader With Document Routing
        """
    )

    pdf_upload = gr.File(
        label="Upload a PDF",
        file_types=[".pdf"],
        type="filepath"
    )

    status = gr.Textbox(
        label="PDF Processing Status",
        value="No PDF uploaded.",
        interactive=False
    )

    pdf_upload.upload(
        fn=process_pdf,
        inputs=pdf_upload,
        outputs=status,
        show_progress="full"
    )

    gr.ChatInterface(
        fn=process_query,
        title="Document Chat",
        description="Ask questions about the uploaded PDF.",
        save_history=True
    )

    download_button = gr.Button("Create JSON Text File")

    download_file = gr.File(
        label="Download JSON Results",
        interactive=False
    )

    download_button.click(
        fn=create_json_download,
        inputs=None,
        outputs=download_file
    )

demo.queue()
demo.launch(show_error=True, debug=True) #will print any errors to Colab

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4a83d2905ce6d30a2a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Text-based PDF detected.
Tagging page 1...
Tagging page 2...
Tagging page 3...
Page 1 | Tags: ['cytiva', 'äkta_ready_flow_kits', 'storage_conditions', 'temperature', 'mike_toner', 'product_management']
Summary: Cytiva communicates recommended storage temperatures for ÄKTA ready flow kits (High Flow Kit F, Modified and High Flow Gradient C, Modified) and potential risks of extended storage below +5°C. Mike Toner of Cytiva provides this information in a letter.
Page 2 | Tags: ["'cytiva'", "'äkta_ready_flow_kits'", "'product_manufacturing'", "'quality_assurance'", "'storage_conditions'", "'temperature'", "'regulatory_compliance'"]
Summary: This document certifies the manufacturing of a Cytiva ÄKTA ready Gradient Flow Section With Inlets, detailing production specifications, storage conditions, and regulatory compliance.
Page 3 | Tags: ["['cytiva'", "'äkta_ready_flow_kits'", "'product_manufacturing'", "'quality_assurance'", "'storage_conditions'", "'temperature'", "'regulatory_compliance'"

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Sliding-window index: 11 chunks

Total PDF processing time: 21.03 seconds
Text-based PDF detected.
Tagging page 1...
Tagging page 2...
Tagging page 3...
Page 1 | Tags: ['cytiva', 'äkta_ready_flow_kits', 'storage_conditions', 'temperature', 'mike_toner', 'product_management']
Summary: Cytiva communicates recommended storage temperature and potential risks for ÄKTA ready flow kits (High Flow Kit F, Modified and High Flow Gradient C, Modified), with operating temperature range of +2C to +40C. Mike Toner, Product Manager, shares this information in a letter.
Page 2 | Tags: ["['cytiva'", "'äkta_ready_flow_kits'", "'product_manufacturing'", "'quality_assurance'", "'storage_conditions'", "'temperature']"]
Summary: Cytiva's ISO 9001 certified quality assurance document for ÄKTA ready Gradient Flow Section With Inlets, detailing manufacturing compliance, storage conditions, and expiration date.
Page 3 | Tags: ["'cytiva'", "'äkta_ready_flow_kits'", "'product_manufacturing'", "'quality_assurance'

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Sliding-window index: 11 chunks

Total PDF processing time: 21.25 seconds
Text-based PDF detected.
Tagging page 1...
Tagging page 2...
Tagging page 3...
Page 1 | Tags: ['cytiva', 'äkta_ready_flow_kits', 'storage_conditions', 'temperature', 'mike_toner', 'product_management']
Summary: Cytiva communicates recommended storage temperature for ÄKTA ready flow kits (High Flow Kit F, Modified and High Flow Gradient C, Modified) as +5°C or above, with an operating temperature range of +2°C to +40°C. Mike Toner, Product Manager, shares this information in a letter.
Page 2 | Tags: ["'cytiva'", "'äkta_ready_flow_kits'", "'product_manufacturing'", "'quality_assurance'", "'storage_conditions'", "'temperature'", "'regulatory_compliance'"]
Summary: This document certifies the manufacturing of a Cytiva ÄKTA ready Gradient Flow Section with Inlets, adhering to ISO 9001 and USP <88> standards. Storage conditions and temperature requirements are specified.
Page 3 | Tags: ["['cytiva'", "'äkta_ready_flow_k

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Sliding-window index: 11 chunks

Total PDF processing time: 22.45 seconds


DEBUG:bm25s:Building index from IDs objects


Raw routing response:  1,2,3

Selected pages: [1, 2, 3]
Page routing ranking:
Rank 1: Page 1
Rank 2: Page 2
Rank 3: Page 3

--- FINAL RETRIEVAL RANKING ---
Rank 1 | Page 1 | Fusion score: 0.032787
Preview: 3 June, 2022Re: ÄKTATMready Flow Kit Storage ConditionsTo Whom It May Concern,The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.3 of the Operating Instruct
Rank 2 | Page 1 | Fusion score: 0.032002
Preview: to brittleness or cracking of the plastic connectors. However, the operating temperature of ÄKTA ready flow kits is +2C to +40C. If the kits are allowed to acclimate to a warmer temperature before bei
Rank 3 | Page 3 | Fusion score: 0.031498
Preview: Product Description: Low Flow Kit, AKTA ready Expiration Date: 20260315 Product Release Criteria We hereby certify that the defined product has been manufactured to meet its specifications and have be
Rank 4 | Page 2 | Fusion score: 0.031010
Preview: Product: ÄKTA™ ready Gradient Flow Sectio